In [1]:
# Installer les dépendances
!pip install pdfplumber pandas openpyxl xlsxwriter


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Users\Etudiant\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


# Extraction des variables

## Library

In [12]:
"""
Extracteur automatique de variables EU-SILC
Utilise pdfplumber + pandas pour extraire et formater les variables
Extraction avec filtrage des en-têtes, pieds de page et table des matières
"""

import pdfplumber
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

## 1. Data EU-SILC

In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-


class EUSILCVariableExtractor:
    """Classe pour extraire les variables EU-SILC depuis un PDF"""
    
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.variables = []
        self.current_variable = None
        
        # Patterns pour filtrer les lignes parasites
        self.filter_patterns = [
            # En-têtes de page
            r'eurostat',
            r'EU-SILC:\s*Methodological\s+guidelines',
            r'–\d{4}\s+Operation',
            
            # Pieds de page avec types de fichier
            r'Personal\s+Register\s+\(R-file\)',
            r'Household\s+Data\s+\(H-file\)',
            r'Household\s+Register\s+\(D-file\)',
            r'Personal\s+Data\s+\(P-file\)',
            
            # Numéros de page seuls
            r'^\d{1,3}$',
            
            # Lignes de séparation
            r'^[_\-=]{3,}$',
        ]
        
        # Compiler les patterns
        self.filter_regex = [re.compile(pattern, re.IGNORECASE) for pattern in self.filter_patterns]
    
    def should_filter_line(self, line):
        """Détermine si une ligne doit être filtrée (en-tête, pied de page, etc.)"""
        line_stripped = line.strip()
        
        # Ignorer les lignes vides
        if not line_stripped:
            return True
        
        # Ignorer les lignes très courtes (< 3 caractères)
        if len(line_stripped) < 3:
            return True
        
        # Vérifier les patterns de filtrage
        for pattern in self.filter_regex:
            if pattern.search(line_stripped):
                return True
        
        return False
    
    def is_table_of_contents_page(self, text):
        """Détecte si une page fait partie de la table des matières"""
        # Indicateurs de table des matières
        toc_indicators = [
            'TABLE OF CONTENTS',
            'CONTENTS',
            'Table of contents',
            'LIST OF VARIABLES',
        ]
        
        # Vérifier si le texte contient des indicateurs TOC
        for indicator in toc_indicators:
            if indicator in text:
                return True
        
        # Vérifier si la page contient beaucoup de points de suite (.......)
        # typique des tables des matières
        dot_lines = text.count('........')
        if dot_lines > 5:
            return True
        
        return False
    
    def is_variable_header(self, line):
        """
        Détecte si une ligne est un en-tête de variable
        """
        line_stripped = line.strip()
        
        # Ne pas traiter si c'est une ligne filtrée
        if self.should_filter_line(line):
            return None
        
        # Pattern pour variables duales avec slash (ex: HY040G/HY040N:)
        dual_slash_pattern = r'^([A-Z]{2}\d{3}G)/([A-Z]{2}\d{3}N):\s*(.+)$'
        match = re.match(dual_slash_pattern, line_stripped)
        if match:
            return ('dual_slash', match)
        
        # Patterns normaux
        patterns = [
            # Variables RG_Z#
            r'^(RG_Z#):\s*(.+)$',
            
            # Variables PMH###
            r'^(PMH\d{3}):\s*(.+)$',
            
            # Variables PHD##
            r'^(PHD\d{2,}):\s*(.+)$',
            
            # Pattern standard (inclut les duales classiques)
            r'^([A-Z]{2}\d{3}[A-Z]?):\s*(.+)$',
        ]
        
        for pattern in patterns:
            match = re.match(pattern, line_stripped)
            if match:
                return ('normal', match)
        
        return None
    
    def is_dual_variable(self, var_code):
        """Vérifie si une variable fait partie d'une paire G/N"""
        dual_pattern = r'^([A-Z]{2}\d{3})([GN])$'
        match = re.match(dual_pattern, var_code)
        
        if match:
            base_code = match.group(1)
            variant = match.group(2)
            return base_code, variant
        
        return None, None
    
    def categorize_variable(self, var_code):
        """Catégorise la variable selon son type"""
        # Variables RG_Z#
        if re.match(r'^RG_Z#$', var_code):
            return 'RG_Z_SERIES'
        
        # Variables PMH###
        if re.match(r'^PMH\d{3}$', var_code):
            return 'PMH_SERIES'
        
        # Variables PHD##
        if re.match(r'^PHD\d{2,}$', var_code):
            return 'PHD_SERIES'
        
        # Variables duales (G/N)
        base_code, variant = self.is_dual_variable(var_code)
        if base_code and variant:
            return f'DUAL_{variant}'
        
        # Variables standards
        if var_code.startswith('DB'):
            return 'D_FILE'
        elif var_code.startswith(('HB', 'HH', 'HS', 'HC', 'HY', 'HI', 'HD')):
            return 'H_FILE'
        elif var_code.startswith(('RB', 'RL', 'RG', 'RX')):
            return 'R_FILE'
        elif var_code.startswith(('PB', 'PY', 'PL', 'PX', 'PH', 'PM')):
            return 'P_FILE'
        
        return 'OTHER'
    
    def extract_variable_info(self, lines, start_idx):
        """Extrait toutes les informations d'une variable"""
        
        # Vérifier le type de header
        header_result = self.is_variable_header(lines[start_idx])
        if not header_result:
            return None
        
        header_type, match = header_result
        
        # Si c'est une variable duale avec slash (HY040G/HY040N)
        if header_type == 'dual_slash':
            return self.extract_dual_slash_variables(lines, start_idx, match)
        else:
            return self.extract_single_variable(lines, start_idx, match)
    
    def extract_dual_slash_variables(self, lines, start_idx, match):
        """
        Extrait les informations pour les variables duales avec slash
        Crée DEUX entrées séparées avec la MÊME description
        """
        var_code_g = match.group(1)  # ex: HY040G
        var_code_n = match.group(2)  # ex: HY040N
        var_name = match.group(3).strip()
        
        # Extraire les métadonnées communes (une seule fois)
        metadata = self.extract_metadata(lines, start_idx)
        
        # Créer deux variables distinctes avec les mêmes données
        variables = []
        
        for var_code in [var_code_g, var_code_n]:
            variable = {
                'variable_code': var_code,
                'variable_name': var_name,
                'variable_category': self.categorize_variable(var_code),
                'base_code': '',
                'variant': '',
                'topic': metadata['topic'],
                'variable_type': metadata['variable_type'],
                'unit': metadata['unit'],
                'reference_period': metadata['reference_period'],
                'mode_of_collection': metadata['mode_of_collection'],
                'in_use_period': metadata['in_use_period'],
                'values_format': metadata['values_format'],
                'flags': metadata['flags'],
                'description': metadata['description'],
                'file_type': ''
            }
            
            # Déterminer le code de base et la variante
            base_code, variant = self.is_dual_variable(var_code)
            if base_code:
                variable['base_code'] = base_code
                variable['variant'] = variant
            
            # Déterminer le type de fichier
            variable['file_type'] = self.determine_file_type(var_code)
            
            variables.append(variable)
        
        return variables
    
    def extract_single_variable(self, lines, start_idx, match):
        """Extrait les informations pour une variable simple"""
        variable = {
            'variable_code': match.group(1),
            'variable_name': match.group(2).strip(),
            'variable_category': '',
            'base_code': '',
            'variant': '',
            'topic': '',
            'variable_type': '',
            'unit': '',
            'reference_period': '',
            'mode_of_collection': '',
            'in_use_period': '',
            'values_format': '',
            'flags': '',
            'description': '',
            'file_type': ''
        }
        
        # Catégoriser
        variable['variable_category'] = self.categorize_variable(variable['variable_code'])
        
        # Pour les variables duales, extraire le code de base
        base_code, variant = self.is_dual_variable(variable['variable_code'])
        if base_code:
            variable['base_code'] = base_code
            variable['variant'] = variant
        
        # Extraire les métadonnées
        metadata = self.extract_metadata(lines, start_idx)
        variable.update(metadata)
        
        # Déterminer le type de fichier
        variable['file_type'] = self.determine_file_type(variable['variable_code'])
        
        return [variable]
    
    def extract_metadata(self, lines, start_idx):
        """Extrait les métadonnées d'une variable en filtrant les lignes parasites"""
        metadata = {
            'topic': '',
            'variable_type': '',
            'unit': '',
            'reference_period': '',
            'mode_of_collection': '',
            'in_use_period': '',
            'values_format': '',
            'flags': '',
            'description': ''
        }
        
        description_lines = []
        values_lines = []
        flags_lines = []
        
        in_values_section = False
        in_flags_section = False
        in_description_section = False
        
        for i in range(start_idx + 1, len(lines)):
            line = lines[i].strip()
            
            # Filtrer les lignes parasites
            if self.should_filter_line(line):
                continue
            
            # Arrêter si on rencontre une nouvelle variable
            header_result = self.is_variable_header(line)
            if header_result:
                break
            
            if not line:
                continue
            
            # Extraction des champs - CORRECTION: extraire sur la même ligne après ":"
            if 'Topic and detailed topic:' in line:
                # Extraire ce qui suit les deux points
                parts = line.split('Topic and detailed topic:', 1)
                if len(parts) > 1:
                    metadata['topic'] = parts[1].strip()
            elif 'Variable type:' in line:
                parts = line.split('Variable type:', 1)
                if len(parts) > 1:
                    metadata['variable_type'] = parts[1].strip()
            elif 'Unit:' in line:
                parts = line.split('Unit:', 1)
                if len(parts) > 1:
                    metadata['unit'] = parts[1].strip()
            elif 'Reference period:' in line:
                parts = line.split('Reference period:', 1)
                if len(parts) > 1:
                    metadata['reference_period'] = parts[1].strip()
            elif 'Mode of collection:' in line:
                parts = line.split('Mode of collection:', 1)
                if len(parts) > 1:
                    metadata['mode_of_collection'] = parts[1].strip()
            elif 'In use (period):' in line:
                parts = line.split('In use (period):', 1)
                if len(parts) > 1:
                    metadata['in_use_period'] = parts[1].strip()
            elif 'VALUES AND FORMAT' in line:
                in_values_section = True
                in_flags_section = False
                in_description_section = False
                continue
            elif line == 'FLAGS':
                in_values_section = False
                in_flags_section = True
                in_description_section = False
                continue
            elif line == 'DESCRIPTION':
                in_values_section = False
                in_flags_section = False
                in_description_section = True
                continue
            
            # Collecter les lignes (en filtrant)
            if in_values_section and line and not self.should_filter_line(line):
                values_lines.append(line)
            elif in_flags_section and line and not self.should_filter_line(line):
                flags_lines.append(line)
            elif in_description_section and line and not self.should_filter_line(line):
                description_lines.append(line)
        
        # Assembler les sections
        metadata['values_format'] = ' '.join(values_lines)
        metadata['flags'] = ' '.join(flags_lines)
        metadata['description'] = ' '.join(description_lines)
        
        return metadata

    
    def determine_file_type(self, var_code):
        """Détermine le type de fichier pour une variable"""
        if var_code.startswith('DB'):
            return 'D-File (Household Register)'
        elif var_code.startswith(('HB', 'HH', 'HS')):
            return 'H-File (Household Data)'
        elif re.match(r'^RG_Z#$', var_code):
            return 'R-File (Personal Register - Special)'
        elif var_code.startswith(('RB', 'RL', 'RG')):
            return 'R-File (Personal Register)'
        elif var_code.startswith('HC'):
            return 'H-File (Household Energy Efficiency)'
        elif var_code.startswith('HY'):
            return 'H-File (Household Income)'
        elif var_code.startswith('HI'):
            return 'H-File (Household Income - Additional)'
        elif var_code.startswith('HD'):
            return 'H-File (Household Data - Additional)'
        elif re.match(r'^PMH\d{3}$', var_code):
            return 'P-File (Personal Health)'
        elif re.match(r'^PHD\d{2,}$', var_code):
            return 'P-File (Personal Health - Detailed)'
        elif var_code.startswith(('PB', 'PY', 'PL', 'PX')):
            return 'P-File (Personal Data)'
        else:
            return 'Unknown'
    
    def extract_all_variables(self):
        """Extrait toutes les variables du PDF"""
        print(f"📄 Ouverture du fichier: {self.pdf_path}")
        
        with pdfplumber.open(self.pdf_path) as pdf:
            print(f"📊 Nombre de pages: {len(pdf.pages)}")
            
            all_text = []
            toc_pages = 0
            
            # Extraire le texte de toutes les pages
            for page_num, page in enumerate(pdf.pages, 1):
                if page_num % 50 == 0:
                    print(f"   Traitement page {page_num}/{len(pdf.pages)}...")
                
                text = page.extract_text()
                if text:
                    # Ignorer les pages de table des matières
                    if self.is_table_of_contents_page(text):
                        toc_pages += 1
                        continue
                    
                    all_text.append(text)
            
            print(f"   Pages de table des matières ignorées: {toc_pages}")
            
            # Joindre tout le texte
            full_text = '\n'.join(all_text)
            lines = full_text.split('\n')
            
            # Filtrer les lignes parasites
            filtered_lines = [line for line in lines if not self.should_filter_line(line)]
            
            print(f"✅ Extraction terminée:")
            print(f"   • Lignes totales: {len(lines)}")
            print(f"   • Lignes filtrées (en-têtes/pieds): {len(lines) - len(filtered_lines)}")
            print(f"   • Lignes utiles: {len(filtered_lines)}")
            print(f"🔍 Recherche des variables...")
            
            # Compteurs
            var_count = 0
            dual_slash_count = 0
            
            # Parcourir les lignes filtrées
            i = 0
            while i < len(filtered_lines):
                header_result = self.is_variable_header(filtered_lines[i])
                
                if header_result:
                    header_type, match = header_result
                    
                    # Extraire la ou les variables
                    variables = self.extract_variable_info(filtered_lines, i)
                    
                    if variables:
                        for variable in variables:
                            self.variables.append(variable)
                            var_count += 1
                            
                            # Afficher avec info (tous les 20 variables)
                            if var_count % 20 == 0:
                                category_info = ""
                                if header_type == 'dual_slash':
                                    category_info = f" [DUAL SLASH]"
                                elif variable['base_code']:
                                    category_info = f" [Dual {variable['variant']}]"
                                elif variable['variable_category'] in ['RG_Z_SERIES', 'PMH_SERIES', 'PHD_SERIES']:
                                    category_info = f" [{variable['variable_category']}]"
                                
                                print(f"   ✓ {var_count} variables extraites... (dernier: {variable['variable_code']}{category_info})")
                            
                            if header_type == 'dual_slash' and variable['variant'] == 'G':
                                dual_slash_count += 1
                
                i += 1
            
            print(f"\n✅ Extraction terminée:")
            print(f"   • Total: {len(self.variables)} variables")
            print(f"   • Variables duales avec slash: {dual_slash_count} paires")
        
        return self.variables
    
    def create_dataframe(self):
        """Crée un DataFrame pandas avec les variables"""
        if not self.variables:
            self.extract_all_variables()
        
        df = pd.DataFrame(self.variables)
        
        columns_order = [
            'file_type',
            'variable_code',
            'variable_name',
            'variable_category',
            'base_code',
            'variant',
            'topic',
            'variable_type',
            'unit',
            'reference_period',
            'mode_of_collection',
            'in_use_period',
            'values_format',
            'flags',
            'description'
        ]
        
        df = df[columns_order]
        
        return df
    
    def export_to_excel(self, output_path='eusilc_variables.xlsx'):
        """Exporte les variables vers un fichier Excel formaté"""
        df = self.create_dataframe()
        
        print(f"\n💾 Création du fichier Excel: {output_path}")
        
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            # Feuille principale
            df.to_excel(writer, sheet_name='All Variables', index=False)
            
            # Feuilles par type de fichier
            d_file = df[df['file_type'].str.contains('D-File', na=False)]
            h_file = df[df['file_type'].str.contains('H-File', na=False)]
            r_file = df[df['file_type'].str.contains('R-File', na=False)]
            p_file = df[df['file_type'].str.contains('P-File', na=False)]
            
            if not d_file.empty:
                d_file.to_excel(writer, sheet_name='D-File', index=False)
            if not h_file.empty:
                h_file.to_excel(writer, sheet_name='H-File', index=False)
            if not r_file.empty:
                r_file.to_excel(writer, sheet_name='R-File', index=False)
            if not p_file.empty:
                p_file.to_excel(writer, sheet_name='P-File', index=False)
            
            # Feuilles spéciales
            dual_vars = df[df['variable_category'].str.contains('DUAL', na=False)]
            if not dual_vars.empty:
                dual_vars.to_excel(writer, sheet_name='Dual Variables (G-N)', index=False)
            
            rg_z_vars = df[df['variable_category'] == 'RG_Z_SERIES']
            if not rg_z_vars.empty:
                rg_z_vars.to_excel(writer, sheet_name='RG_Z Series', index=False)
            
            pmh_vars = df[df['variable_category'] == 'PMH_SERIES']
            if not pmh_vars.empty:
                pmh_vars.to_excel(writer, sheet_name='PMH Series', index=False)
            
            phd_vars = df[df['variable_category'] == 'PHD_SERIES']
            if not phd_vars.empty:
                phd_vars.to_excel(writer, sheet_name='PHD Series', index=False)
            
            # Formatage
            workbook = writer.book
            header_format = workbook.add_format({
                'bold': True,
                'bg_color': '#4472C4',
                'font_color': 'white',
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'text_wrap': True
            })
            
            for sheet_name in writer.sheets:
                worksheet = writer.sheets[sheet_name]
                worksheet.set_row(0, 30)
                
                # Largeurs
                worksheet.set_column('A:A', 30)
                worksheet.set_column('B:B', 15)
                worksheet.set_column('C:C', 40)
                worksheet.set_column('D:D', 20)
                worksheet.set_column('E:E', 12)
                worksheet.set_column('F:F', 10)
                worksheet.set_column('G:G', 30)
                worksheet.set_column('H:H', 20)
                worksheet.set_column('I:I', 15)
                worksheet.set_column('J:J', 20)
                worksheet.set_column('K:K', 20)
                worksheet.set_column('L:L', 25)
                worksheet.set_column('M:M', 40)
                worksheet.set_column('N:N', 30)
                worksheet.set_column('O:O', 80)
                
                # En-têtes
                for col_num, value in enumerate(df.columns.values):
                    worksheet.write(0, col_num, value, header_format)
                
                worksheet.freeze_panes(1, 0)
                worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        
        print(f"✅ Fichier Excel créé: {len(df)} variables")
    
    def export_to_csv(self, output_path='eusilc_variables.csv'):
        """Exporte vers CSV"""
        df = self.create_dataframe()
        df.to_csv(output_path, index=False, encoding='utf-8-sig', sep=';')
        print(f"✅ Fichier CSV créé: {output_path}")
    
    def export_to_markdown(self, output_path='eusilc_variables.md'):
        """Exporte vers Markdown"""
        df = self.create_dataframe()
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(f"# EU-SILC Variables Documentation\n\n")
            f.write(f"*Généré le: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
            f.write(f"**Total: {len(df)} variables**\n\n")
            
            # Statistiques
            dual_vars = df[df['variable_category'].str.contains('DUAL', na=False)]
            if not dual_vars.empty:
                f.write(f"### Variables Duales: {len(dual_vars)}\n\n")
        
        print(f"✅ Fichier Markdown créé: {output_path}")
    
    def print_summary(self):
        """Affiche un résumé"""
        df = self.create_dataframe()
        
        print("\n" + "="*70)
        print("📊 RÉSUMÉ")
        print("="*70)
        print(f"\n✅ Total: {len(df)} variables")
        
        # Par type
        print("\n📁 Par type de fichier:")
        file_counts = df['file_type'].value_counts()
        for file_type, count in file_counts.items():
            print(f"   • {file_type}: {count}")
        
        # Duales
        dual_vars = df[df['variable_category'].str.contains('DUAL', na=False)]
        if not dual_vars.empty:
            print(f"\n🔄 Variables duales: {len(dual_vars)}")
            print(f"   • G: {len(dual_vars[dual_vars['variant'] == 'G'])}")
            print(f"   • N: {len(dual_vars[dual_vars['variant'] == 'N'])}")
        
        print("\n" + "="*70)


def main():
    """Fonction principale"""
    print("="*70)
    print("🔍 EU-SILC Variable Extractor - OPTIMISÉ")
    print("="*70)
    print("\n✨ Améliorations:")
    print("   ✓ Filtrage des en-têtes de page (eurostat, etc.)")
    print("   ✓ Filtrage des pieds de page (R-file, H-file, etc.)")
    print("   ✓ Exclusion de la table des matières")
    print("   ✓ Support HY040G/HY040N, RG_Z#, PMH###, PHD##")
    print("="*70 + "\n")
    
    pdf_path = "Methodological guidelines 2023 operation v6 (accessible).pdf"
    
    if not Path(pdf_path).exists():
        print(f"❌ Fichier introuvable!")
        return
    
    extractor = EUSILCVariableExtractor(pdf_path)
    extractor.extract_all_variables()
    extractor.print_summary()
    
    print("\n💾 Exportation...")
    extractor.export_to_excel('eusilc_optimized.xlsx')
    extractor.export_to_csv('eusilc_optimized.csv')
    extractor.export_to_markdown('eusilc_optimized.md')

    print("\n✅ Terminé!")
    print("\n📂 Fichiers:")
    print("   • eusilc_optimized.xlsx")
    print("   • eusilc_optimized.csv")
    print("   • eusilc_optimized.md")


if __name__ == "__main__":
    main()

🔍 EU-SILC Variable Extractor - OPTIMISÉ

✨ Améliorations:
   ✓ Filtrage des en-têtes de page (eurostat, etc.)
   ✓ Filtrage des pieds de page (R-file, H-file, etc.)
   ✓ Exclusion de la table des matières
   ✓ Support HY040G/HY040N, RG_Z#, PMH###, PHD##

📄 Ouverture du fichier: Methodological guidelines 2023 operation v6 (accessible).pdf
📊 Nombre de pages: 649


KeyboardInterrupt: 

## 2. Extraction Data EU-LFS

In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EU-LFS Variable Extractor - Version Finale
Extraction automatique des variables du Labour Force Survey depuis PDF
Format: "Description VARIABLE_CODE" sur la même ligne
"""

import pdfplumber
import pandas as pd
import re
from pathlib import Path


class EULFSVariableExtractor:
    """
    Extracteur spécialisé pour EU Labour Force Survey
    Format: Description et code de variable sur la même ligne
    """
    
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.variables = []
        self.current_section = None
        
        # Sections du document LFS
        self.sections = {
            'core': 'Core variables',
            'derived_labour': 'Derived variables for standard labour market analyses',
            'derived_household': 'Derived household variables',
            'former': 'Former and formerly derived variables'
        }
    
    def detect_section(self, text):
        """Détecte dans quelle section du document on se trouve"""
        if 'a. Core variables (overview)' in text:
            return 'core'
        elif 'b. Derived variables for standard labour market' in text:
            return 'derived_labour'
        elif 'c. Derived household variables' in text:
            return 'derived_household'
        elif 'd. Former and formerly derived variables' in text:
            return 'former'
        return None
    
    def extract_variable_from_line(self, line):
        """
        Extrait variable(s) depuis une ligne
        Format attendu: "Description text CODE" ou "Description text * CODE" 
        ou "Description text * CODE1, CODE2"
        Retourne une liste de variables (peut être plusieurs si codes multiples)
        """
        line = line.strip()
        
        # Ignorer les lignes trop courtes
        if len(line) < 10:
            return []
        
        # Ignorer les en-têtes et titres (mais PAS "Labour status" seul car WSTATOR suit)
        ignore_patterns = [
            'Description Variable name',
            'Demographic background',
            'Employment characteristics',
            'Atypical work',
            'Hours worked',
            'Second job',
            'Previous work experience',
            'Search for employment',
            'Methods used',
            'Education and training',
            'Main labour status',
            'Situation one year',
            'Income',
            'Technical items',
            'OVERVIEW',
            'LIST OF VARIABLES'
        ]
        
        # Ne pas ignorer si la ligne contient juste le titre ET un code après
        if not any(code_pattern in line for code_pattern in [r'[A-Z]{3,}']):
            if any(pattern in line for pattern in ignore_patterns):
                return []
        
        # Patterns pour extraction
        patterns = [
            # Pattern 1: Description * CODE1, CODE2 (codes multiples avec *)
            r'^(.+?)\s+\*\s+([A-Z][A-Z0-9]{2,14}(?:,\s*[A-Z][A-Z0-9]{2,14})+)$',
            
            # Pattern 2: Description CODE1, CODE2 (codes multiples sans *)
            r'^(.+?)\s+([A-Z][A-Z0-9]{2,14}(?:,\s*[A-Z][A-Z0-9]{2,14})+)$',
            
            # Pattern 3: Description * CODE (code unique avec *)
            r'^(.+?)\s+\*\s+([A-Z][A-Z0-9]{2,14})$',
            
            # Pattern 4: Description CODE (code unique sans *)
            r'^(.+?)\s+([A-Z][A-Z0-9]{2,14})$',
        ]
        
        for pattern in patterns:
            match = re.match(pattern, line)
            if match:
                description = match.group(1).strip()
                var_codes = match.group(2).strip()
                
                # Vérifier que la description est assez longue
                if len(description) < 5:
                    continue
                
                # Vérifier que le code n'est pas un mot commun par erreur
                common_words = ['THE', 'AND', 'FOR', 'WITH', 'FROM', 'THIS', 'THAT', 'WORK']
                if var_codes in common_words:
                    continue
                
                # Séparer les codes multiples
                codes_list = [code.strip() for code in var_codes.split(',')]
                
                # Créer une variable pour chaque code
                results = []
                for code in codes_list:
                    if code:  # Vérifier que le code n'est pas vide
                        results.append({
                            'variable_code': code,
                            'description': description
                        })
                
                return results
        
        return []
    
    def extract_from_pages(self, start_page, end_page):
        """
        Extrait les variables depuis une plage de pages
        """
        with pdfplumber.open(self.pdf_path) as pdf:
            
            for page_num in range(start_page, min(end_page + 1, len(pdf.pages) + 1)):
                page = pdf.pages[page_num - 1]
                text = page.extract_text()
                
                if not text:
                    continue
                
                # Détecter changement de section
                section_detected = self.detect_section(text)
                if section_detected:
                    self.current_section = section_detected
                
                lines = text.split('\n')
                
                for line in lines:
                    variables = self.extract_variable_from_line(line)
                    
                    # variables est maintenant une liste
                    for variable in variables:
                        # Vérifier que ce n'est pas un doublon
                        if not any(v['variable_code'] == variable['variable_code'] 
                                 for v in self.variables):
                            variable['section'] = self.current_section
                            self.variables.append(variable)
    
    def generate_short_description(self, description):
        """Génère une description courte depuis la description complète"""
        short = description[:150]
        
        if len(description) > 150:
            last_space = short.rfind(' ')
            if last_space > 100:
                short = short[:last_space]
            short += '...'
        
        return short
    
    def generate_long_description(self, variable_data):
        """Génère une documentation structurée"""
        sections_list = []
        
        var_code = variable_data.get('variable_code', '')
        description = variable_data.get('description', '')
        
        sections_list.append(f"**{var_code}**")
        sections_list.append(f"**Description:** {description}")
        
        section = variable_data.get('section', '')
        if section:
            section_name = self.sections.get(section, section)
            sections_list.append(f"**Section:** {section_name}")
        
        var_category = variable_data.get('variable_category', '')
        if var_category:
            sections_list.append(f"**Category:** {var_category}")
        
        return "\n\n".join(sections_list)
    
    def categorize_variable(self, var_code):
        """Catégorise la variable selon son préfixe"""
        # Gestion des codes multiples (ex: "NACE3D, NA113D")
        first_code = var_code.split(',')[0].strip()
        
        if first_code.startswith('HH'):
            return 'HOUSEHOLD'
        elif first_code.startswith(('FT', 'PT')):
            return 'EMPLOYMENT_TYPE'
        elif first_code.startswith('HW'):
            return 'WORKING_TIME'
        elif first_code.startswith(('LOOK', 'SEEK')):
            return 'JOB_SEARCH'
        elif first_code.startswith(('EXIST', 'PREV', 'YEARPR', 'MONTHPR')):
            return 'WORK_HISTORY'
        elif first_code.startswith('METHOD'):
            return 'JOB_SEARCH_METHOD'
        elif first_code.startswith(('STAPRO', 'NACE', 'ISCO', 'NA', 'IS')):
            return 'CLASSIFICATION'
        elif first_code.startswith(('INCOME', 'COEFF', 'INCDECIL')):
            return 'INCOME_WEIGHT'
        elif first_code.startswith(('HAT', 'EDUC', 'ISCED', 'COUR')):
            return 'EDUCATION'
        elif first_code.startswith(('COUNTRY', 'REGION', 'COUNTR', 'NATION')):
            return 'GEOGRAPHY'
        elif first_code.startswith(('REF', 'INT', 'QUEST', 'QUARTER', 'YEAR', 'REM')):
            return 'TECHNICAL'
        elif first_code.startswith(('WSTAT', 'MAINSTAT', 'ILOSTAT')):
            return 'LABOUR_STATUS'
        elif first_code.startswith(('AGE', 'SEX', 'MARSTAT', 'DATEBIR', 'YEARBIR')):
            return 'DEMOGRAPHIC'
        elif first_code.startswith(('TEMP', 'PERM')):
            return 'CONTRACT_TYPE'
        elif first_code.startswith(('AVAIL', 'WANT')):
            return 'AVAILABILITY'
        elif first_code.startswith(('SIGN', 'NOWK')):
            return 'WORK_STATUS'
        elif first_code.startswith(('EVEN', 'NIGHT', 'SAT', 'SUN', 'SHIFT')):
            return 'WORK_SCHEDULE'
        elif first_code.startswith(('SUPE', 'SUPER', 'SUPV')):
            return 'SUPERVISION'
        elif first_code.startswith('SIZE'):
            return 'FIRM_SIZE'
        elif first_code.startswith(('HOME', 'PLACE')):
            return 'WORK_LOCATION'
        elif first_code.startswith(('START', 'YSTART', 'MSTART', 'LEAVE', 'DURU')):
            return 'JOB_TENURE'
        elif first_code.startswith('PROXY'):
            return 'SURVEY_MODE'
        elif first_code.startswith('DEGURBA'):
            return 'URBANISATION'
        elif first_code.startswith('ESEG'):
            return 'SOCIO_ECONOMIC'
        elif first_code.startswith('WISH'):
            return 'WORK_PREFERENCES'
        elif first_code.startswith('WAY'):
            return 'WORK_PREFERENCES'
        elif first_code.startswith('NEED'):
            return 'CARE_FACILITIES'
        elif first_code.startswith('REGIST'):
            return 'REGISTRATION'
        elif first_code.startswith('PRESE'):
            return 'PREVIOUS_SITUATION'
        elif first_code.startswith('HOUR'):
            return 'WORKING_TIME'
        elif first_code.startswith('QHHNUM'):
            return 'HOUSEHOLD'
        else:
            return 'OTHER'
    
    def extract_all_variables(self):
        """Extrait toutes les variables du PDF LFS"""
        print(f"📄 Ouverture du fichier LFS: {self.pdf_path}")
        
        with pdfplumber.open(self.pdf_path) as pdf:
            print(f"📊 Nombre de pages: {len(pdf.pages)}")
        
        # Extraire toutes les sections overview (pages 4-9)
        print("\n🔍 Extraction des variables...")
        self.extract_from_pages(4, 9)
        
        print(f"\n✅ Extraction terminée: {len(self.variables)} variables brutes")
        
        # Compléter les métadonnées
        print("📝 Enrichissement des métadonnées...")
        for var in self.variables:
            var['variable_category'] = self.categorize_variable(var['variable_code'])
            var['short_description'] = self.generate_short_description(var['description'])
            var['long_description'] = self.generate_long_description(var)
            var['variable_type'] = ''
            var['periodicity'] = ''
            var['codes'] = ''
            var['remarks'] = ''
        
        print(f"✅ {len(self.variables)} variables enrichies")
        
        return self.variables
    
    def create_dataframe(self):
        """Crée un DataFrame pandas"""
        if not self.variables:
            self.extract_all_variables()
        
        df = pd.DataFrame(self.variables)
        
        columns_order = [
            'variable_code',
            'description',
            'short_description',
            'long_description',
            'section',
            'variable_category',
            'variable_type',
            'periodicity',
            'codes',
            'remarks'
        ]
        
        for col in columns_order:
            if col not in df.columns:
                df[col] = ''
        
        return df[columns_order]
    
    def export_to_excel(self, output_path='eulfs_variables.xlsx'):
        """Exporte vers Excel avec feuilles multiples"""
        df = self.create_dataframe()
        
        print(f"\n💾 Création du fichier Excel: {output_path}")
        
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            # Feuille principale
            df.to_excel(writer, sheet_name='All Variables', index=False)
            
            # Feuilles par section
            for section_key, section_name in self.sections.items():
                section_df = df[df['section'] == section_key]
                if not section_df.empty:
                    sheet_name = section_name[:31]
                    section_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Feuilles par catégorie (top 10)
            categories = df['variable_category'].value_counts().head(10)
            for category in categories.index:
                if category and category != 'OTHER':
                    cat_df = df[df['variable_category'] == category]
                    if not cat_df.empty:
                        sheet_name = category[:31]
                        cat_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Formatage
            workbook = writer.book
            header_format = workbook.add_format({
                'bold': True,
                'bg_color': '#2E75B6',
                'font_color': 'white',
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'text_wrap': True
            })
            
            for sheet_name in writer.sheets:
                worksheet = writer.sheets[sheet_name]
                worksheet.set_row(0, 30)
                
                # Largeurs de colonnes
                worksheet.set_column('A:A', 20)
                worksheet.set_column('B:B', 65)
                worksheet.set_column('C:C', 55)
                worksheet.set_column('D:D', 85)
                worksheet.set_column('E:E', 22)
                worksheet.set_column('F:F', 28)
                worksheet.set_column('G:G', 15)
                worksheet.set_column('H:H', 15)
                worksheet.set_column('I:I', 40)
                worksheet.set_column('J:J', 40)
                
                # En-têtes
                for col_num, value in enumerate(df.columns.values):
                    worksheet.write(0, col_num, value, header_format)
                
                worksheet.freeze_panes(1, 0)
                worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        
        print(f"✅ Fichier Excel créé: {len(df)} variables")
    
    def export_to_csv(self, output_path='eulfs_variables.csv'):
        """Exporte vers CSV"""
        df = self.create_dataframe()
        df.to_csv(output_path, index=False, encoding='utf-8-sig', sep=';')
        print(f"✅ Fichier CSV créé: {output_path}")
    
    def print_summary(self):
        """Affiche un résumé"""
        df = self.create_dataframe()
        
        print("\n" + "="*70)
        print("📊 RÉSUMÉ EU-LFS")
        print("="*70)
        print(f"\n✅ Total: {len(df)} variables")
        
        print("\n📁 Par section:")
        section_counts = df['section'].value_counts()
        for section, count in section_counts.items():
            section_name = self.sections.get(section, section)
            print(f"   • {section_name}: {count}")
        
        print("\n📂 Par catégorie (top 15):")
        category_counts = df['variable_category'].value_counts()
        for category, count in category_counts.head(15).items():
            print(f"   • {category}: {count}")
        
        # Afficher quelques exemples
        print("\n📋 Exemples de variables extraites:")
        for i, row in df.head(10).iterrows():
            print(f"\n   {row['variable_code']}:")
            print(f"   → {row['description'][:80]}...")
            print(f"   → Catégorie: {row['variable_category']}")
        
        print("\n" + "="*70)


def main():
    """Fonction principale"""
    print("="*70)
    print("🔍 EU-LFS Variable Extractor - Version Finale")
    print("="*70)
    print("\n✨ Extraction depuis le User Guide Eurostat")
    print("="*70 + "\n")
    
    pdf_path = "EULFS-Database-UserGuide_eurostat.pdf"
    
    if not Path(pdf_path).exists():
        print(f"❌ Fichier introuvable: {pdf_path}")
        return
    
    extractor = EULFSVariableExtractor(pdf_path)
    extractor.extract_all_variables()
    extractor.print_summary()
    
    print("\n💾 Exportation des résultats...")
    extractor.export_to_excel('eulfs_variables.xlsx')
    extractor.export_to_csv('eulfs_variables.csv')

    print("\n✅ Terminé! Fichiers disponibles dans le répertoire courant.")
if __name__ == "__main__":
    main()

🔍 EU-LFS Variable Extractor - Version Finale

✨ Extraction depuis le User Guide Eurostat

📄 Ouverture du fichier LFS: EULFS-Database-UserGuide_eurostat.pdf
📊 Nombre de pages: 76

🔍 Extraction des variables...

✅ Extraction terminée: 155 variables brutes
📝 Enrichissement des métadonnées...
✅ 155 variables enrichies

📊 RÉSUMÉ EU-LFS

✅ Total: 155 variables

📁 Par section:
   • Core variables: 107
   • Derived variables for standard labour market analyses: 24
   • Derived household variables: 16
   • Former and formerly derived variables: 8

📂 Par catégorie (top 15):
   • HOUSEHOLD: 25
   • CLASSIFICATION: 25
   • JOB_SEARCH_METHOD: 13
   • EDUCATION: 11
   • TECHNICAL: 10
   • OTHER: 9
   • WORKING_TIME: 7
   • GEOGRAPHY: 7
   • JOB_SEARCH: 6
   • WORK_SCHEDULE: 5
   • DEMOGRAPHIC: 4
   • JOB_TENURE: 4
   • WORK_HISTORY: 4
   • LABOUR_STATUS: 3
   • INCOME_WEIGHT: 3

📋 Exemples de variables extraites:

   HHSEQNUM:
   → Sequence number in the household...
   → Catégorie: HOUSEHOLD

   HH

## 3. Data Household Budget Survey (HBS)

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
HBS Variable Extractor
Extraction automatique des variables du Household Budget Survey depuis PDF
"""

class HBSVariableExtractor:
    """
    Extracteur spécialisé pour Household Budget Survey (HBS)
    Gère les record layouts avec structure tabulaire complexe
    """
    
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.variables = []
        self.current_file_type = None  # 'household' or 'member'
        
        # Catégories de variables HBS
        self.variable_categories = {
            'HA': 'HOUSEHOLD_IDENTIFICATION',
            'HB': 'HOUSEHOLD_COMPOSITION',
            'HC': 'HOUSEHOLD_SOCIOECONOMIC',
            'HD': 'HOUSEHOLD_ACTIVITY',
            'HE': 'HOUSEHOLD_EXPENDITURE',
            'HI': 'HOUSEHOLD_INCOME',
            'HJ': 'EXPENDITURE_ABROAD',
            'HQ': 'HOUSEHOLD_QUANTITIES',
            'MA': 'MEMBER_IDENTIFICATION',
            'MB': 'MEMBER_DEMOGRAPHIC',
            'MC': 'MEMBER_EDUCATION',
            'ME': 'MEMBER_EMPLOYMENT',
            'MF': 'MEMBER_INCOME',
            'EUR_': 'EXPENDITURE_EURO'
        }
    
    def detect_file_type(self, text):
        """Détecte le type de fichier (household ou member)"""
        if 'Household record layout' in text:
            return 'household'
        elif 'Household member record layout' in text:
            return 'member'
        return None
    
    def categorize_variable(self, var_code):
        """Catégorise la variable selon son préfixe"""
        for prefix, category in self.variable_categories.items():
            if var_code.startswith(prefix):
                return category
        return 'OTHER'
    
    def clean_text(self, text):
        """Nettoie le texte des caractères parasites"""
        if not text:
            return ''
        # Remplacer les retours à la ligne multiples
        text = re.sub(r'\n+', ' ', text)
        # Retirer les espaces multiples
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def extract_from_table(self, page):
        """Extrait les variables depuis les tables du PDF avec une approche robuste"""
        page_text = page.extract_text()
        
        # Détecter le type de fichier
        file_type = self.detect_file_type(page_text)
        if file_type:
            self.current_file_type = file_type
        
        tables = page.extract_tables()
        variables_found = []
        
        for table in tables:
            if not table or len(table) < 1:
                continue
            
            # Parser chaque ligne de la table
            for row in table:
                if not row:
                    continue
                
                # Convertir la ligne en liste de chaînes
                cells = []
                for cell in row:
                    if cell:
                        cell_text = self.clean_text(str(cell))
                        cells.append(cell_text)
                    else:
                        cells.append('')
                
                # Skip empty rows
                if not any(cells):
                    continue
                
                # Chercher un code de variable
                # Les codes variables commencent par des lettres majuscules et sont courts
                var_code_pattern = r'^([A-Z][A-Z0-9_\.]{1,20})$'
                
                var_code = None
                var_code_idx = -1
                
                # Chercher le code variable dans toutes les cellules (généralement cellule 0 ou 1)
                for i, cell in enumerate(cells[:4]):
                    if cell and re.match(var_code_pattern, cell):
                        # Vérifier que ce n'est pas un en-tête
                        if cell not in ['VARIABLE', 'NAME', 'LABEL', 'FORMAT', 'NOTES', 
                                       'RULES', 'POSSIBLE', 'VALUES', 'MEASUREMENT', 'UNIT', 
                                       'ANONYMISATION', 'ACTIONS', 'INT', 'CHA', 'DEC']:
                            var_code = cell
                            var_code_idx = i
                            break
                
                if not var_code:
                    continue
                
                # Extraire les informations restantes
                var_label = ''
                possible_values = ''
                var_format = ''
                notes = ''
                
                # Les cellules après le code contiennent les autres infos
                remaining_cells = cells[var_code_idx + 1:]
                
                # Parcourir les cellules restantes
                for i, cell in enumerate(remaining_cells):
                    if not cell:
                        continue
                    
                    # Détecter le format (INT, CHA, DEC)
                    if re.match(r'^(INT|CHA|DEC)\s*\d*\.?\d*$', cell):
                        if not var_format:
                            var_format = cell
                    # Détecter les notes/actions (mots-clés spécifiques)
                    elif any(keyword in cell for keyword in ['Dropped', 'suppressed', 'replaced', 
                                                               'Recoded', 'Variable to be used',
                                                               'Top coding', 'merged', 'ROUNDING',
                                                               'ROUNDED', 'Recomputed']):
                        notes = notes + ' ' + cell if notes else cell
                    # Sinon, c'est probablement une description ou des valeurs possibles
                    else:
                        # Si on n'a pas encore de description, c'est la description
                        if not var_label:
                            var_label = cell
                        # Sinon, c'est probablement les valeurs possibles
                        elif not possible_values:
                            # Vérifier si c'est une liste de valeurs (contient des chiffres)
                            if re.search(r'\d+', cell) or 'not specified' in cell.lower():
                                possible_values = cell
                            else:
                                # Ajouter à la description
                                var_label = var_label + ' ' + cell
                
                # Créer la variable
                variable = {
                    'variable_code': var_code,
                    'description': var_label,
                    'possible_values': possible_values,
                    'format': var_format,
                    'notes': notes.strip(),
                    'file_type': self.current_file_type,
                    'variable_category': self.categorize_variable(var_code)
                }
                
                variables_found.append(variable)
        
        return variables_found
    
    def extract_all_variables(self):
        """Extrait toutes les variables du PDF HBS"""
        print(f"📄 Ouverture du fichier HBS: {self.pdf_path}")
        
        with pdfplumber.open(self.pdf_path) as pdf:
            print(f"📊 Nombre de pages: {len(pdf.pages)}")
            
            var_count = 0
            
            # Commencer à la page 11 où commencent les record layouts
            start_page = 10  # Index 10 = page 11
            
            for page_num in range(start_page, len(pdf.pages)):
                page = pdf.pages[page_num]
                
                if (page_num - start_page) % 5 == 0:
                    print(f"   Traitement page {page_num + 1}/{len(pdf.pages)}...")
                
                variables = self.extract_from_table(page)
                
                for variable in variables:
                    # Vérifier les doublons
                    if not any(v['variable_code'] == variable['variable_code'] for v in self.variables):
                        # Générer descriptions
                        variable['short_description'] = self.generate_short_description(variable)
                        variable['long_description'] = self.generate_long_description(variable)
                        
                        self.variables.append(variable)
                        var_count += 1
                        
                        if var_count % 50 == 0:
                            print(f"   ✓ {var_count} variables extraites...")
            
            print(f"\n✅ Extraction terminée: {len(self.variables)} variables")
        
        return self.variables
    
    def generate_short_description(self, variable):
        """Génère une description courte"""
        desc = variable.get('description', '')
        if len(desc) > 150:
            last_space = desc[:150].rfind(' ')
            if last_space > 100:
                desc = desc[:last_space] + '...'
            else:
                desc = desc[:150] + '...'
        return desc
    
    def generate_long_description(self, variable):
        """Génère une documentation structurée"""
        sections = []
        
        var_code = variable.get('variable_code', '')
        description = variable.get('description', '')
        
        sections.append(f"**{var_code}**")
        sections.append(f"**Description:** {description}")
        
        # Type de fichier
        file_type = variable.get('file_type', '')
        if file_type:
            file_label = 'Household Level' if file_type == 'household' else 'Member Level'
            sections.append(f"**Level:** {file_label}")
        
        # Catégorie
        category = variable.get('variable_category', '')
        if category:
            sections.append(f"**Category:** {category}")
        
        # Format
        var_format = variable.get('format', '')
        if var_format:
            sections.append(f"**Format:** {var_format}")
        
        # Valeurs possibles
        possible_values = variable.get('possible_values', '')
        if possible_values and len(possible_values) > 5:
            sections.append(f"**Possible Values:** {possible_values}")
        
        # Notes
        notes = variable.get('notes', '')
        if notes and len(notes) > 5:
            sections.append(f"**Notes:** {notes}")
        
        return "\n\n".join(sections)
    
    def create_dataframe(self):
        """Crée un DataFrame pandas"""
        if not self.variables:
            self.extract_all_variables()
        
        df = pd.DataFrame(self.variables)
        
        # Ordre des colonnes
        columns_order = [
            'variable_code',
            'description',
            'short_description',
            'long_description',
            'file_type',
            'variable_category',
            'format',
            'possible_values',
            'notes'
        ]
        
        # S'assurer que toutes les colonnes existent
        for col in columns_order:
            if col not in df.columns:
                df[col] = ''
        
        return df[columns_order]
    
    def export_to_excel(self, output_path='hbs_variables.xlsx'):
        """Exporte vers Excel avec feuilles multiples"""
        df = self.create_dataframe()
        
        print(f"\n💾 Création du fichier Excel: {output_path}")
        
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            # Feuille principale
            df.to_excel(writer, sheet_name='All Variables', index=False)
            
            # Feuille par type de fichier
            for file_type in df['file_type'].unique():
                if file_type and file_type != 'None':
                    type_df = df[df['file_type'] == file_type]
                    sheet_name = f"{file_type.capitalize()} Variables"[:31]
                    type_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Feuilles par catégorie principale
            categories = df['variable_category'].value_counts()
            for category in categories.index:
                if category and category != 'OTHER':
                    cat_df = df[df['variable_category'] == category]
                    if len(cat_df) >= 3:  # Au moins 3 variables
                        sheet_name = category[:31]
                        cat_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Formatage
            workbook = writer.book
            header_format = workbook.add_format({
                'bold': True,
                'bg_color': '#4472C4',
                'font_color': 'white',
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'text_wrap': True
            })
            
            for sheet_name in writer.sheets:
                worksheet = writer.sheets[sheet_name]
                worksheet.set_row(0, 30)
                
                # Largeurs de colonnes
                worksheet.set_column('A:A', 20)  # variable_code
                worksheet.set_column('B:B', 50)  # description
                worksheet.set_column('C:C', 40)  # short_description
                worksheet.set_column('D:D', 80)  # long_description
                worksheet.set_column('E:E', 15)  # file_type
                worksheet.set_column('F:F', 25)  # variable_category
                worksheet.set_column('G:G', 12)  # format
                worksheet.set_column('H:H', 50)  # possible_values
                worksheet.set_column('I:I', 40)  # notes
                
                # En-têtes
                for col_num, value in enumerate(df.columns.values):
                    worksheet.write(0, col_num, value, header_format)
                
                worksheet.freeze_panes(1, 0)
                worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        
        print(f"✅ Fichier Excel créé: {len(df)} variables")
    
    def export_to_csv(self, output_path='hbs_variables.csv'):
        """Exporte vers CSV"""
        df = self.create_dataframe()
        df.to_csv(output_path, index=False, encoding='utf-8-sig', sep=';')
        print(f"✅ Fichier CSV créé: {output_path}")
    
    def print_summary(self):
        """Affiche un résumé"""
        df = self.create_dataframe()
        
        print("\n" + "="*70)
        print("📊 RÉSUMÉ HBS (Household Budget Survey)")
        print("="*70)
        print(f"\n✅ Total: {len(df)} variables")
        
        print("\n📁 Par type de fichier:")
        type_counts = df['file_type'].value_counts()
        for file_type, count in type_counts.items():
            if file_type:
                print(f"   • {file_type.capitalize()}: {count}")
        
        print("\n📂 Par catégorie:")
        category_counts = df['variable_category'].value_counts()
        for category, count in category_counts.head(20).items():
            print(f"   • {category}: {count}")
        
        print("\n" + "="*70)


def main():
    """Fonction principale"""
    print("="*70)
    print("🔍 HBS Variable Extractor")
    print("   Household Budget Survey 2010 - Scientific-use files")
    print("="*70)
    print("\n✨ Extraction des record layouts (Household & Member)")
    print("="*70 + "\n")
    
    # Chemin du fichier
    pdf_path = "HBS User Manual 2010.pdf"
    
    if not Path(pdf_path).exists():
        print(f"❌ Fichier introuvable: {pdf_path}")
        return
    
    extractor = HBSVariableExtractor(pdf_path)
    extractor.extract_all_variables()
    extractor.print_summary()
    
    print("\n💾 Exportation...")
    extractor.export_to_excel()
    extractor.export_to_csv()
    
    print("\n✅ Terminé!")
    print("\n📂 Fichiers créés dans /mnt/user-data/outputs/:")
    print("   • hbs_variables.xlsx")
    print("   • hbs_variables.csv")


if __name__ == "__main__":
    main()

🔍 HBS Variable Extractor
   Household Budget Survey 2010 - Scientific-use files

✨ Extraction des record layouts (Household & Member)

📄 Ouverture du fichier HBS: HBS User Manual 2010.pdf
📊 Nombre de pages: 56
   Traitement page 11/56...
   Traitement page 16/56...
   ✓ 50 variables extraites...
   ✓ 100 variables extraites...
   Traitement page 21/56...
   ✓ 150 variables extraites...
   Traitement page 26/56...
   ✓ 200 variables extraites...
   ✓ 250 variables extraites...
   Traitement page 31/56...
   ✓ 300 variables extraites...
   Traitement page 36/56...
   ✓ 350 variables extraites...
   Traitement page 41/56...
   ✓ 400 variables extraites...
   Traitement page 46/56...
   Traitement page 51/56...
   Traitement page 56/56...

✅ Extraction terminée: 434 variables

📊 RÉSUMÉ HBS (Household Budget Survey)

✅ Total: 434 variables

📁 Par type de fichier:
   • Household: 412
   • Member: 22

📂 Par catégorie:
   • EXPENDITURE_EURO: 387
   • HOUSEHOLD_COMPOSITION: 14
   • MEMBER_EMPLO

## 4. Data HFCS (Core and Non-core Variables)

In [19]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
HFCS Variable Extractor - Adapted for Core and Non-core Variables
Extraction from HFCS User Database Documentation PDFs
"""

import re
import pdfplumber
import pandas as pd
from pathlib import Path
from datetime import datetime


class HFCSVariableExtractor:
    """Class to extract HFCS variables from PDF documentation"""
    
    def __init__(self, pdf_paths):
        """
        Initialize with list of PDF paths
        pdf_paths: list of tuples (path, doc_type) where doc_type is 'core' or 'non-core'
        """
        self.pdf_paths = pdf_paths
        self.variables = []
        
        # Patterns to filter out page headers/footers
        self.filter_patterns = [
            r'HFCS Documentation',
            r'^\d{1,3}$',  # Page numbers alone
            r'^[_\-=]{3,}$',  # Separator lines
            r'Core and derived variables',
            r'Non-core variables',
            r'2021 Wave',
            r'July 2023',
        ]
        
        self.filter_regex = [re.compile(pattern, re.IGNORECASE) for pattern in self.filter_patterns]
    
    def should_filter_line(self, line):
        """Determine if a line should be filtered (headers, footers, etc.)"""
        line_stripped = line.strip()
        
        if not line_stripped or len(line_stripped) < 2:
            return True
        
        for pattern in self.filter_regex:
            if pattern.search(line_stripped):
                return True
        
        return False
    
    def is_table_of_contents(self, text):
        """Detect if page is table of contents"""
        toc_indicators = [
            'Contents',
            'TABLE OF CONTENTS',
        ]
        
        for indicator in toc_indicators:
            if indicator in text:
                return True
        
        # Check for many dot leaders (common in TOC)
        if text.count('...') > 5:
            return True
        
        # Check for pattern like "Variable_name page_number" repeated many times
        # This is typical of TOC pages
        lines = text.split('\n')
        toc_pattern_count = 0
        for line in lines:
            # Pattern: text followed by 2-digit number at end
            if re.match(r'^.+\s+\d{1,3}$', line.strip()):
                toc_pattern_count += 1
        
        if toc_pattern_count > 10:  # If more than 10 lines match TOC pattern
            return True
        
        return False
    
    def is_variable_header(self, line):
        """
        Detect if a line is a variable header
        HFCS format: VARIABLE_CODE variable name
        Examples: 
        - SA0100 survey year
        - RNA0200 Citizenship
        - DA1000SH Real assets as share of gross wealth
        - DA1000i Has real assets (derived)
        - HNB130$x HMR mortgage $x: institution you have loan with
        - HB320$x$y other property $x mortgage $y
        - PNA0600x Education of father/mother
        
        EXCLUDE:
        - HB301$loops (variables with 'loops')
        """
        line_stripped = line.strip()
        
        if self.should_filter_line(line):
            return None
        
        # EXCLUDE variables containing "loops"
        if 'loops' in line_stripped.lower():
            return None
        
        # Pattern for HFCS variables with various suffixes:
        # - Base: 2-4 capital letters + 3-4 digits
        # - Suffix options:
        #   * SH, i, G, N (letter suffixes, can be uppercase or lowercase)
        #   * $x, $y (single index)
        #   * $x$y (double index)
        #   * x (simple x suffix)
        # Pattern breakdown:
        # [A-Z]{1,4} = 1-4 capital letters
        # \d{3,4} = 3-4 digits
        # (?:...) = non-capturing group for suffixes
        # [A-Za-z]{1,2}|\$[xy](?:\$[xy])?|x = suffixes: 1-2 letters (case insensitive) OR $x/$y/$x$y OR x
        
        pattern = r'^([A-Z]{1,4}\d{3,4}(?:[A-Za-z]{1,2}|\$[xy](?:\$[xy])?)?)\s+(.+)$'
        
        match = re.match(pattern, line_stripped)
        if match:
            var_code = match.group(1)
            var_name = match.group(2)
            
            # Additional validation: exclude if variable name contains "loops"
            if 'loops' in var_name.lower():
                return None
            
            # Variable name should be reasonable length
            # Accept both lowercase and uppercase start
            if len(var_name) > 2:
                return ('variable', var_code, var_name)
        
        return None
    
    def extract_variable_info(self, lines, start_idx, doc_type):
        """Extract all information for a variable"""
        
        header_result = self.is_variable_header(lines[start_idx])
        if not header_result:
            return None
        
        _, var_code, var_name = header_result
        
        variable = {
            'document_type': doc_type,
            'variable_code': var_code,
            'variable_name': var_name,
            'question_text': '',
            'reference_unit': '',
            'reference_period': '',
            'coding': '',
            'filtering': '',
            'survey_definition': '',
            'technical_definition': '',
            'notes': ''
        }
        
        # Extract metadata from following lines
        current_section = None
        section_lines = {
            'question': [],
            'reference_unit': [],
            'reference_period': [],
            'coding': [],
            'loop': [],
            'filtering': [],
            'survey_definition': [],
            'technical_definition': [],
            'notes': []
        }
        
        i = start_idx + 1
        while i < len(lines):
            line = lines[i].strip()
            
            # Stop if we hit another variable
            if self.is_variable_header(line):
                break
            
            if not line or self.should_filter_line(line):
                i += 1
                continue
            
            # Detect sections
            if line.startswith('Reference unit:'):
                current_section = 'reference_unit'
                content = line.replace('Reference unit:', '').strip()
                if content:
                    section_lines['reference_unit'].append(content)
            elif line.startswith('Reference period:'):
                current_section = 'reference_period'
                content = line.replace('Reference period:', '').strip()
                if content:
                    section_lines['reference_period'].append(content)
            elif line.startswith('Coding:'):
                current_section = 'coding'
                content = line.replace('Coding:', '').strip()
                if content:
                    section_lines['coding'].append(content)
            elif line.startswith('Loop:'):
                current_section = 'loop'
                content = line.replace('Loop:', '').strip()
                if content:
                    section_lines['loop'].append(content)
            elif line.startswith('Filtering:'):
                current_section = 'filtering'
                content = line.replace('Filtering:', '').strip()
                if content:
                    section_lines['filtering'].append(content)
            elif line == 'Survey definition:':
                current_section = 'survey_definition'
            elif line == 'Technical definition:':
                current_section = 'technical_definition'
            elif current_section == 'survey_definition':
                section_lines['survey_definition'].append(line)
            elif current_section == 'technical_definition':
                section_lines['technical_definition'].append(line)
            elif current_section == 'reference_unit':
                section_lines['reference_unit'].append(line)
            elif current_section == 'reference_period':
                section_lines['reference_period'].append(line)
            elif current_section == 'coding':
                section_lines['coding'].append(line)
            elif current_section == 'filtering':
                section_lines['filtering'].append(line)
            elif current_section is None:
                # This is likely the question text
                section_lines['question'].append(line)
            
            i += 1
        
        # Assemble sections
        variable['question_text'] = ' '.join(section_lines['question'])
        variable['reference_unit'] = ' '.join(section_lines['reference_unit'])
        variable['reference_period'] = ' '.join(section_lines['reference_period'])
        variable['coding'] = ' '.join(section_lines['coding'])
        variable['filtering'] = ' '.join(section_lines['filtering'])
        variable['survey_definition'] = ' '.join(section_lines['survey_definition'])
        variable['technical_definition'] = ' '.join(section_lines['technical_definition'])
        
        # Add loop info to notes if present
        if section_lines['loop']:
            variable['notes'] = 'Loop: ' + ' '.join(section_lines['loop'])
        
        return variable
    
    def extract_from_pdf(self, pdf_path, doc_type):
        """Extract all variables from a single PDF"""
        print(f"\n📄 Processing: {Path(pdf_path).name}")
        print(f"   Type: {doc_type.upper()}")
        
        variables_found = []
        
        with pdfplumber.open(pdf_path) as pdf:
            print(f"   Pages: {len(pdf.pages)}")
            
            all_text = []
            toc_pages = 0
            start_extraction = False
            
            # Define the starting markers for each document type
            # For non-core, we need both markers to ensure we're at the right place
            start_markers = {
                'core': [('1 Core variables', None), ('Core variables', 'Sample information')],
                'non-core': [('Non-core variables', '1 Demographics')]
            }
            
            # Extract text from all pages
            for page_num, page in enumerate(pdf.pages, 1):
                if page_num % 50 == 0:
                    print(f"   Processing page {page_num}/{len(pdf.pages)}...")
                
                text = page.extract_text()
                if text:
                    # Skip table of contents
                    if self.is_table_of_contents(text):
                        toc_pages += 1
                        continue
                    
                    # Check if we've reached the start of the content section
                    if not start_extraction:
                        for marker_tuple in start_markers.get(doc_type, []):
                            if len(marker_tuple) == 2:
                                primary_marker, secondary_marker = marker_tuple
                            else:
                                primary_marker = marker_tuple[0]
                                secondary_marker = None
                            
                            # Check if primary marker is present
                            if primary_marker in text:
                                # If secondary marker is specified, check for it too
                                if secondary_marker is None or secondary_marker in text:
                                    start_extraction = True
                                    print(f"   ✓ Found start marker: '{primary_marker}' on page {page_num}")
                                    break
                    
                    # Only add text after we've found the start marker
                    if start_extraction:
                        all_text.append(text)
            
            print(f"   TOC pages skipped: {toc_pages}")
            
            if not start_extraction:
                print(f"   ⚠️  Warning: Start marker not found for {doc_type} document!")
            
            # Join all text and split into lines
            full_text = '\n'.join(all_text)
            lines = full_text.split('\n')
            
            # Filter lines
            filtered_lines = [line for line in lines if not self.should_filter_line(line)]
            
            print(f"   Total lines: {len(lines)}")
            print(f"   Filtered lines: {len(lines) - len(filtered_lines)}")
            print(f"   Useful lines: {len(filtered_lines)}")
            print(f"🔍 Searching for variables...")
            
            # Extract variables
            i = 0
            var_count = 0
            
            while i < len(filtered_lines):
                header_result = self.is_variable_header(filtered_lines[i])
                
                if header_result:
                    variable = self.extract_variable_info(filtered_lines, i, doc_type)
                    
                    if variable:
                        variables_found.append(variable)
                        var_count += 1
                        
                        if var_count % 50 == 0:
                            print(f"   ✓ {var_count} variables extracted... (last: {variable['variable_code']})")
                
                i += 1
            
            print(f"✅ Extracted {len(variables_found)} variables from {doc_type}")
        
        return variables_found
    
    def extract_all_variables(self):
        """Extract variables from all PDFs"""
        print("="*80)
        print("🔍 HFCS Variable Extractor")
        print("="*80)
        
        for pdf_path, doc_type in self.pdf_paths:
            if not Path(pdf_path).exists():
                print(f"❌ File not found: {pdf_path}")
                continue
            
            variables = self.extract_from_pdf(pdf_path, doc_type)
            self.variables.extend(variables)
        
        print(f"\n{'='*80}")
        print(f"✅ TOTAL VARIABLES EXTRACTED: {len(self.variables)}")
        print(f"{'='*80}")
        
        return self.variables
    
    def create_dataframe(self):
        """Create a pandas DataFrame with all variables"""
        if not self.variables:
            self.extract_all_variables()
        
        df = pd.DataFrame(self.variables)
        
        # Reorder columns
        columns_order = [
            'document_type',
            'variable_code',
            'variable_name',
            'question_text',
            'reference_unit',
            'reference_period',
            'coding',
            'filtering',
            'survey_definition',
            'technical_definition',
            'notes'
        ]
        
        df = df[columns_order]
        
        return df
    
    def export_to_excel(self, output_path='hfcs_variables.xlsx'):
        """Export variables to formatted Excel file"""
        df = self.create_dataframe()
        
        print(f"\n💾 Creating Excel file: {output_path}")
        
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            # Main sheet with all variables
            df.to_excel(writer, sheet_name='All Variables', index=False)
            
            # Separate sheets by document type
            core_vars = df[df['document_type'] == 'core']
            non_core_vars = df[df['document_type'] == 'non-core']
            
            if not core_vars.empty:
                core_vars.to_excel(writer, sheet_name='Core Variables', index=False)
            if not non_core_vars.empty:
                non_core_vars.to_excel(writer, sheet_name='Non-Core Variables', index=False)
            
            # Formatting
            workbook = writer.book
            header_format = workbook.add_format({
                'bold': True,
                'bg_color': '#4472C4',
                'font_color': 'white',
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'text_wrap': True
            })
            
            # Format all sheets
            for sheet_name in writer.sheets:
                worksheet = writer.sheets[sheet_name]
                worksheet.set_row(0, 30)
                
                # Set column widths
                worksheet.set_column('A:A', 15)  # document_type
                worksheet.set_column('B:B', 15)  # variable_code
                worksheet.set_column('C:C', 50)  # variable_name
                worksheet.set_column('D:D', 60)  # question_text
                worksheet.set_column('E:E', 20)  # reference_unit
                worksheet.set_column('F:F', 20)  # reference_period
                worksheet.set_column('G:G', 50)  # coding
                worksheet.set_column('H:H', 30)  # filtering
                worksheet.set_column('I:I', 60)  # survey_definition
                worksheet.set_column('J:J', 60)  # technical_definition
                worksheet.set_column('K:K', 40)  # notes
                
                # Apply header format
                for col_num, value in enumerate(df.columns.values):
                    worksheet.write(0, col_num, value, header_format)
                
                # Freeze top row
                worksheet.freeze_panes(1, 0)
                worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        
        print(f"✅ Excel file created: {len(df)} variables")
    
    def export_to_csv(self, output_path='hfcs_variables.csv'):
        """Export to CSV"""
        df = self.create_dataframe()
        df.to_csv(output_path, index=False, encoding='utf-8-sig', sep=';')
        print(f"✅ CSV file created: {output_path}")
    
    def export_to_markdown(self, output_path='hfcs_variables.md'):
        """Export to Markdown"""
        df = self.create_dataframe()
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(f"# HFCS Variables Documentation\n\n")
            f.write(f"*Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
            f.write(f"**Total variables: {len(df)}**\n\n")
            
            # Statistics by document type
            f.write("## Summary\n\n")
            doc_counts = df['document_type'].value_counts()
            for doc_type, count in doc_counts.items():
                f.write(f"- {doc_type.capitalize()} variables: {count}\n")
            
            f.write("\n## Variables\n\n")
            
            # Group by document type
            for doc_type in ['core', 'non-core']:
                subset = df[df['document_type'] == doc_type]
                if not subset.empty:
                    f.write(f"### {doc_type.capitalize()} Variables ({len(subset)})\n\n")
                    
                    for _, row in subset.iterrows():
                        f.write(f"#### {row['variable_code']}: {row['variable_name']}\n\n")
                        if row['question_text']:
                            f.write(f"**Question:** {row['question_text']}\n\n")
                        if row['reference_unit']:
                            f.write(f"**Reference unit:** {row['reference_unit']}\n\n")
                        if row['reference_period']:
                            f.write(f"**Reference period:** {row['reference_period']}\n\n")
                        f.write("---\n\n")
        
        print(f"✅ Markdown file created: {output_path}")
    
    def print_summary(self):
        """Print summary statistics"""
        df = self.create_dataframe()
        
        print("\n" + "="*80)
        print("📊 SUMMARY")
        print("="*80)
        print(f"\n✅ Total variables: {len(df)}")
        
        # By document type
        print("\n📁 By document type:")
        doc_counts = df['document_type'].value_counts()
        for doc_type, count in doc_counts.items():
            print(f"   • {doc_type.capitalize()}: {count}")
        
        print("\n" + "="*80)


def main():
    """Main function"""
    print("="*80)
    print("🔍 HFCS Variable Extractor")
    print("="*80)
    print("\n✨ Features:")
    print("   ✓ Extracts from Core and Non-Core variable documents")
    print("   ✓ Filters headers, footers, and TOC")
    print("   ✓ Captures question text, coding, definitions")
    print("="*80 + "\n")
    
    # Define PDF paths
    pdf_paths = [
        ("UDB_w4.2 - HFCS Core and derived variables - 2021 Wave.pdf", "core"),
        ("UDB_w4.3 - HFCS Non-core variables - 2021 Wave.pdf", "non-core")
    ]
    
    # Create extractor and process
    extractor = HFCSVariableExtractor(pdf_paths)
    extractor.extract_all_variables()
    extractor.print_summary()
    
    # Export to all formats
    print("\n💾 Exporting to all formats...")
    extractor.export_to_excel('hfcs_variables.xlsx')
    extractor.export_to_csv('hfcs_variables.csv')
    extractor.export_to_markdown('hfcs_variables.md')

    print("\n✅ Complete!")
    print("\n📂 Files created:")
    print("   • hfcs_variables.xlsx")
    print("   • hfcs_variables.csv")
    print("   • hfcs_variables.md")


if __name__ == "__main__":
    main()

🔍 HFCS Variable Extractor

✨ Features:
   ✓ Extracts from Core and Non-Core variable documents
   ✓ Filters headers, footers, and TOC
   ✓ Captures question text, coding, definitions

🔍 HFCS Variable Extractor

📄 Processing: UDB_w4.2 - HFCS Core and derived variables - 2021 Wave.pdf
   Type: CORE
   Pages: 201
   ✓ Found start marker: '1 Core variables' on page 21
   Processing page 50/201...
   Processing page 100/201...
   Processing page 150/201...
   Processing page 200/201...
   TOC pages skipped: 18
   Total lines: 5940
   Filtered lines: 366
   Useful lines: 5574
🔍 Searching for variables...
   ✓ 50 variables extracted... (last: HB2300)
   ✓ 100 variables extracted... (last: HC070$x)
   ✓ 150 variables extracted... (last: PE0100d)
   ✓ 200 variables extracted... (last: PG0510)
   ✓ 250 variables extracted... (last: DA1120)
   ✓ 300 variables extracted... (last: DA2108)
   ✓ 350 variables extracted... (last: DI1510)
   ✓ 400 variables extracted... (last: DL1120b)
✅ Extracted 442 